In [ ]:
import pandas as pd
from Bio import SeqIO
import torch
print(torch.cuda.device_count())

2


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)  # None if this is a CPU-only build
print(torch.cuda.is_available())

2.3.0
12.1
True


In [ ]:
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_name(1))

NVIDIA RTX A4000
NVIDIA RTX A4000


## Turn Interaction Data into sequences for input to MINT

### Get sequences for target proteins from uniprot FASTA file

In [ ]:
def load_uniprot_sequences(fasta_file):
    """
    Args:
        fasta_file: Path to the UniProt FASTA file containing human proteome sequences.
ra
    Returns:
        A dictionary mapping UniProt IDs to their corresponding protein sequences.
        
    """
    seq_dict = {}
    for record in SeqIO.parse(fasta_file,"fasta"):
        parts = record.id.split('|')
        if len(parts) >=2:
            uniprot_id = parts[1]
            seq_dict[uniprot_id] = str(record.seq)
    return seq_dict

def map_sequences_to_uniprot_ids(df, seq_dict):
    """
    Args:
        df: DataFrame containing a column 'uniprot_id' with UniProt IDs.
        seq_dict: Dictionary mapping UniProt IDs to protein sequences.

    Returns:
        A DataFrame with an additional column 'sequence' containing the protein sequences.
    """
    df['sequence'] = df['uniprot_id'].map(seq_dict)
    return df

In [ ]:
fasta = "mint/data/raw/uniprotkb_human_proteome_2025_11_26.fasta"
seq_dict = load_uniprot_sequences(fasta) # Uniprot to sequence dictionary made from fasta with human proteome

In [ ]:
# Read file with the interactions of interest
target_interactions = pd.read_csv("mint/data/raw/AllInteractionsNicholas.csv")
target_interactions.shape

(149416, 2)

In [ ]:
# Sort each pair so (A,B) and (B,A) become the same
target_interactions.columns = ['Protein1', 'Protein2']
target_interactions = target_interactions.dropna(subset=['Protein1', 'Protein2'])  # Drop rows with NaN values

target_interactions['sorted_pair'] = target_interactions.apply(
    lambda row: tuple(sorted([row['Protein1'], row['Protein2']])), axis=1
)

# Drop duplicates
target_interactions_deduped = target_interactions.drop_duplicates(subset='sorted_pair').drop(columns='sorted_pair')

print(f"Before: {len(target_interactions)}")
print(f"After:  {len(target_interactions_deduped)}")
print(f"Removed: {len(target_interactions) - len(target_interactions_deduped)} duplicate pairs.")

Before: 149416
After:  149416
Removed: 0 duplicate pairs.


In [ ]:
# Get unique gene names in target_interactions
unique_genes = pd.unique(target_interactions[["Protein1", "Protein2"]].values.ravel())

In [ ]:
gene_mapping = pd.DataFrame({'uniprot_id' : unique_genes})

In [ ]:
gene_mapping = map_sequences_to_uniprot_ids(gene_mapping, seq_dict)
gene_mapping.head()

,uniprot_id,sequence
0,Q8IZP0,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...
1,Q9NYB9,MAELQMLLEEEIPGGRRALFDSYTNLERVADYCENNYIQSADKQRA...
2,P16333,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...
3,Q8WUW1,MAGQEDPVQREIHQDWANREYIEIITSSIKKIADFLNSFDMSCRSR...
4,Q9Y2V7,MAEGSGEVVAVSATGAANGLNNGAGGTSATTCNPLSRKLHKILETR...


In [ ]:
pairs_df_deduped = pd.DataFrame({
    'Protein1': target_interactions_deduped['Protein1'].map(seq_dict),
    'Protein2': target_interactions_deduped['Protein2'].map(seq_dict),
})

In [ ]:
pairs_df_deduped.head()

,Protein1,Protein2
0,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAELQMLLEEEIPGGRRALFDSYTNLERVADYCENNYIQSADKQRA...
1,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...
2,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAGQEDPVQREIHQDWANREYIEIITSSIKKIADFLNSFDMSCRSR...
3,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAEGSGEVVAVSATGAANGLNNGAGGTSATTCNPLSRKLHKILETR...
4,MAAYSYRPGPGAGPGPAAGAALPDQSFLWNVFQRVDKDRSGVISDT...,MASYPYRQGCPGAAGQAPGAPPGSYYPGPPNSGGQYGSGLPPGGGY...


### Convert STRING's ensembl ID to uniprot ID

---

### Process pair data to get embeddings from MINT

In [ ]:
# Process data to split between GPU 0 and GPU 1

GPU0_BATCHFILE = 'mint/data/gpu0_batch.csv' # Save location for GPU 0 batch file
GPU1_BATCHFILE = 'mint/data/gpu1_batch.csv' # Save location for GPU 1 batch file

# Split the pairs_df_deduped into two halves for GPU processing
mid = len(pairs_df_deduped) // 2
pairs_df_deduped.iloc[:mid].to_csv(GPU0_BATCHFILE, index=False)
pairs_df_deduped.iloc[mid:].to_csv(GPU1_BATCHFILE, index=False)

print(f"GPU 0: {len(pairs_df_deduped.iloc[:mid])} pairs")
print(f"GPU 1: {len(pairs_df_deduped.iloc[mid:])} pairs")

GPU 0: 74708 pairs
GPU 1: 74708 pairs


### Processing complete: Now run the run_mint.py script to get embeddings for each protein in the pairs_df_deduped dataframe. 

The following commands ensure that the script runs in the background and continues running even if the terminal is closed.
You can check the log files to monitor the progress of the script.

`nohup python run_mint.py cuda:0 data/gpu0_batch.csv output/gpu0_output.pt > log0.txt 2>&1 &`

`disown`

`nohup python run_mint.py cuda:1 data/gpu1_batch.csv output/gpu1_output.pt > log1.txt 2>&1 &`
`disown`

### Got embeddings, recombine and map back to original pairs.

In [ ]:
import torch
import pandas as pd
# Load the output embedding files from both GPUs 
GPU_OUTPUTFILE0 = 'mint/output/aram_output0_2.pt'  # Output file from GPU 0
GPU_OUTPUTFILE1 = 'mint/output/aram_output1_2.pt'  # Output file from GPU 1

# Load both embedding files
emb0 = torch.load(GPU_OUTPUTFILE0)
emb1 = torch.load(GPU_OUTPUTFILE1)

print(emb0.shape)  
print(emb1.shape)

# Merge
all_embeddings = torch.cat([emb0, emb1], dim=0)
print(all_embeddings.shape)  
# Load the deduplicated pairs in the same order
print(pairs_df_deduped.shape) # should be half of all_embeddings (sep_chains doubles it)

torch.Size([74491, 2560])
torch.Size([74491, 2560])
torch.Size([148982, 2560])
(148982, 2)


### Get gene mapping from before and original interactions

In [ ]:
pairs_df_deduped.head()

,Protein1,Protein2
0,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAELQMLLEEEIPGGRRALFDSYTNLERVADYCENNYIQSADKQRA...
1,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...
2,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAGQEDPVQREIHQDWANREYIEIITSSIKKIADFLNSFDMSCRSR...
3,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...,MAEGSGEVVAVSATGAANGLNNGAGGTSATTCNPLSRKLHKILETR...
4,MAAYSYRPGPGAGPGPAAGAALPDQSFLWNVFQRVDKDRSGVISDT...,MASYPYRQGCPGAAGQAPGAPPGSYYPGPPNSGGQYGSGLPPGGGY...


In [ ]:
gene_mapping.head()

,uniprot_id,sequence
0,Q8IZP0,MAELQMLLEEEIPSGKRALIESYQNLTRVADYCENNYIQATDKRKA...
1,Q9NYB9,MAELQMLLEEEIPGGRRALFDSYTNLERVADYCENNYIQSADKQRA...
2,P16333,MAEEVVVVAKFDYVAQQEQELDIKKNERLWLLDDSKSWWRVRNSMN...
3,Q8WUW1,MAGQEDPVQREIHQDWANREYIEIITSSIKKIADFLNSFDMSCRSR...
4,Q9Y2V7,MAEGSGEVVAVSATGAANGLNNGAGGTSATTCNPLSRKLHKILETR...


In [ ]:
# Split back into individual sequence embeddings

emb_seq1 = all_embeddings[:, :1280]   # first protein
emb_seq2 = all_embeddings[:, 1280:]   # second protein

# Map back to pairs

target_interactions_deduped['emb_seq1'] = list(emb_seq1.numpy())
target_interactions_deduped['emb_seq2'] = list(emb_seq2.numpy())
target_interactions_deduped.head()
target_interactions_deduped.to_parquet('mint/data/aram_per_interactions_embeddings_2.parquet', index=False)